In [1]:
# Chargement des packages:
import numpy as np
import pandas as pd


In [2]:
## Chargement des données et nettoyage
df_ciqual = pd.read_excel("data/ciqual.xlsx")
df_ciqual.columns = df_ciqual.columns.str.replace("\n", " ").str.strip() ## pour simplifier l'écriture du nom des colonnes avec des espaces qui sont sinon écrit avec des \n

# La base Ciqual contient parfois des tirets pour des informations manquantes, parfois des 0, parfois des seuils (ex: < 0.2: on veut harmoniser cela pour calculer des indices par la suite), parfois "traces" quand la quantité mesurée est trop faible pour être quantifiée
# On utilisera une fonction
def clean_ciqual(val):
    if pd.isna(val) or str(val).strip() in ["-", "traces", ""]:
        return 0.0
    val_clean = str(val).replace(",", ".").replace("<", "")
    try:
        return float(val_clean)
    except:
        return 0.0


## On recherche le nom des colonnes où ces problèmes peuvent apparaître:
print(list(df_ciqual.columns))

## On créé une liste des colonnes uniquement textuelles (ie celles sur lesquelles on appliquera pas notre fonction):
cols_metadata = [
    "alim_grp_code",
    "alim_ssgrp_code",
    "alim_ssssgrp_code",
    "alim_grp_nom_fr",
    "alim_ssgrp_nom_fr",
    "alim_ssssgrp_nom_fr",
    "alim_code",
    "alim_nom_fr",
    "alim_nom_sci",
    "Facteur de Jones"]

# 2. On sélectionne automatiquement toutes les colonnes qui NE SONT PAS des métadonnées
cols_nutrition = [c for c in df_ciqual.columns if c not in cols_metadata]

# 3. On applique la fonction sur chaque colonne nutritionnelle
for col in cols_nutrition:
  df_ciqual[col] = df_ciqual[col].apply(clean_ciqual)

# On vérifie si des valeurs non numériques ou NaN subsistent
valeurs_manquantes = df_ciqual[cols_nutrition].isna().sum().sum()
print(f"{valeurs_manquantes}")

/opt/python/lib/python3.13/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


['alim_grp_code', 'alim_ssgrp_code', 'alim_ssssgrp_code', 'alim_grp_nom_fr', 'alim_ssgrp_nom_fr', 'alim_ssssgrp_nom_fr', 'alim_code', 'alim_nom_fr', 'alim_nom_sci', 'Energie, Règlement UE N° 1169 2011 (kJ 100 g)', 'Energie, Règlement UE N° 1169 2011 (kcal 100 g)', 'Energie, N x facteur Jones, avec fibres (kJ 100 g)', 'Energie, N x facteur Jones, avec fibres (kcal 100 g)', 'Eau (g 100 g)', 'Protéines, N x facteur de Jones (g 100 g)', 'Protéines, N x 6.25 (g 100 g)', 'Glucides (g 100 g)', 'Lipides (g 100 g)', 'Sucres (g 100 g)', 'Fructose (g 100 g)', 'Galactose (g 100 g)', 'Glucose (g 100 g)', 'Lactose (g 100 g)', 'Maltose (g 100 g)', 'Saccharose (g 100 g)', 'Amidon (g 100 g)', 'Fibres alimentaires (g 100 g)', 'Polyols totaux (g 100 g)', 'Cendres (g 100 g)', 'Alcool (éthanol) (g 100 g)', 'Acides organiques (g 100 g)', 'AG saturés (g 100 g)', 'AG monoinsaturés (g 100 g)', 'AG polyinsaturés (g 100 g)', 'AG 4:0, butyrique (g 100 g)', 'AG 6:0, caproïque (g 100 g)', 'AG 8:0, caprylique (g 100

# Définition de scores alimentaires / nutritifs: le score SAIN et le score LIM 


## Le Score SAIN
Le score SAIN est une mesure des nutriments de qualité. Il calcule l'apport de 5 nutriments que l'on peut juger "positifs" pour la santé (Protéines, Fibres, Calcium, Fer, Vitamine C). Ces nutriments sont ramené à 100 kcal / 418,4 kJ (afin de prendre en compte la densité energétique de certains produits: en effet cela évite d'avantager les aliments très denses et gras qui contiendraient mécaniquement plus de vitamines/protéines pour 100g, mais qui apporteraient énormément de calories.) et divisé par leurs Apports Nutritionnels Conseillés (ANC). Nos variables de nutriments étant mesurées en g/100g dans la table Ciqual, il conviendra de les convertir pour leur valeur nutritive pour 418,4 kJ. Pour ce faire on utilisera la règle de 3 suivante: 

$ \text{Nutriment}_{\text{pour 418 kJ}} = \frac{\text{Nutriment}_{\text{en } g/100g}}{\text{Énergie}_{\text{en } kJ/100g}} \times 418,4$

Ensuite, pour calculer le score SAIN il faut diviser l'apport de chaque nutriment par les Apports Nutritionnels Conseillés. Ceux-ci sont, d'après le papier de Marion Tharrey, Matthieu Maillot, Véronique Azaïs-Braesco and Nicole Darmon, de 65 g/418,4 kJ pour les protéines, de 25g/418,4 kJ pour les fibres, de 110 mg/418.4 kJ pour la vitamine C, de 900mg/418,4 kJ pour le calcium et de 12,5 mg/418,4 kJ pour le fer.

La formule générale du score SAIN s'exprime comme la moyenne des ratios d'adéquation des 5 nutriments positifs:

$\text{SAIN} = \frac{1}{5} \sum_{p=1}^{5} \left( \frac{\text{Nutriment}_{p \text{ (pour 418,4 kJ)}}}{\text{ANC}_p} \right) \times 100$

En développant cette formule avec les valeurs numériques des ANC propres à chaque nutriment :  

$\text{SAIN} = \frac{1}{5} \left( \frac{\text{Protéines}_{418,4\text{kJ}}}{65} + \frac{\text{Fibres}_{418,4\text{kJ}}}{25} + \frac{\text{Vitamine C}_{418,4\text{kJ}}}{110} + \frac{\text{Calcium}_{418,4\text{kJ}}}{900} + \frac{\text{Fer}_{418,4\text{kJ}}}{12,5} \right) \times 100$

Conformément à la méthodologie de Darmon et al. (2009), le calcul du SAIN ne plafonne pas individuellement les ratios de chaque nutriment (ce qui est discutable car sans plafonnement individuel, on pourrait avoir un aliment qui contiendrait une quantité importante d'un seul nutriment (par exemple, un jus de fruit contenant 500% de l'ANC en vitamine C, ce seul nutriment donnerait un ratio de 5). À lui seul, il ferait monter le SAIN à un niveau très élevé). Ici on appliquera toutefois un bornage global du score final entre 1 et 100. 

In [3]:
# On renomme la colonne d'énergie dans la table ciqual
col_kJ = "Energie, Règlement UE N° 1169 2011 (kJ 100 g)"

# Gestion des aliments à 0 kcal (ex: eau) pour éviter division par zéro
energie = np.where(df_ciqual[col_kJ] == 0, np.nan, df_ciqual[col_kJ])

# Création des nouvelles colonnes exprimées pour 100 kcal pour nos 5 nutriments d'intérêt
df_ciqual["prot_418kJ"] = np.nan_to_num((df_ciqual["Protéines, N x facteur de Jones (g 100 g)"] / energie) * 418.4)
df_ciqual["fibres_418kJ"] = np.nan_to_num((df_ciqual["Fibres alimentaires (g 100 g)"] / energie) * 418.4)
df_ciqual["calcium_418kJ"] = np.nan_to_num((df_ciqual["Calcium (mg 100 g)"] / energie) * 418.4)
df_ciqual["fer_418kJ"] = np.nan_to_num((df_ciqual["Fer (mg 100 g)"] / energie) * 418.4)
df_ciqual["vitc_418kJ"] = np.nan_to_num((df_ciqual["Vitamine C (mg 100 g)"] / energie) * 418.4)

# Valeurs de référence exactes 
ANC = {"prot": 65, "fibres": 25, "vitc": 110,"calcium": 900,  "fer": 12.5}

# Ratios individuels par nutriment:

r_prot = df_ciqual["prot_418kJ"] / ANC["prot"]
r_fibres = df_ciqual["fibres_418kJ"] / ANC["fibres"]
r_calcium = df_ciqual["calcium_418kJ"] / ANC["calcium"]
r_fer = df_ciqual["fer_418kJ"] / ANC["fer"]
r_vitc = df_ciqual["vitc_418kJ"] / ANC["vitc"]

# 5. Calcul du Score SAIN (moyenne des 5 ratios * 100)
sain_brut = ((r_prot + r_fibres + r_vitc + r_calcium + r_fer) / 5.0) * 100.0
df_ciqual["SAIN"] = np.clip(sain_brut, 1.0, 100.0)

## Le Score LIM

Le score LIM est le pendant négatif du SAIN : il mesure la teneur de l'aliment en nutriments dont la surconsommation est associée à un risque accru de problèmes cardiovasculaires. Il prend en compte 3 nutriments à limiter : les Acides Gras Saturés (AGS), les sucres ajoutés (ou sucres) et le sodium. Contrairement au SAIN qui est calculé sur une base d'énergie (418,4 kJ/100 kcal) pour évaluer la densité nutritionnelle, le score LIM est exprimé pour 100g de produit. Ce choix méthodologique repose sur le fait que les seuils d'exposition maximale pour les nutriments néfastes sont définis par rapport à la masse totale d'aliments ingérés quotidiennement. 

Pour calculer le score LIM, la teneur en chaque nutriment à limiter (exprimée pour 100g) est rapportée à sa Valeur Maximale Recommandée (VMR) journalière. D'après le papier de référence (Darmon et al., 2009), ces seuils maximaux sont de :22g pour les Acides Gras Saturés, de 50g pour les sucres ajoutés et de 3153mg pour le sodium. La formule mathématique du score LIM pour un aliment se définit ainsi :

$\text{LIM} = \frac{1}{3} \left( \frac{\text{AGS}_{\text{g/100g}}}{22} + \frac{\text{Sucres}_{\text{g/100g}}}{50} + \frac{\text{Sodium}_{\text{mg/100g}}}{3153} \right) \times 100$

On bornera également le score LIM à 100.

In [4]:
# Calcul du score LIM (plus simple à calculer car pas besoin de conversion)
r_ags = df_ciqual["AG saturés (g 100 g)"] / 22.0  
r_sucres = df_ciqual["Sucres (g 100 g)"] / 50.0
r_sodium = df_ciqual["Sodium (mg 100 g)"] / 3157.0

lim_brut = ((r_ags + r_sucres + r_sodium) / 3.0) * 100.0
df_ciqual["LIM"] = np.clip(lim_brut, 0.1, 100.0)